**Libraries**

In [1]:
from singleCAM_IROS._pipeline_support import _handle_dirpaths

from typing import Callable, NamedTuple
from pathlib import Path

import numpy as np
from numpy.typing import NDArray 
from scipy.optimize import least_squares

from bloodmoon.coords import pos2shift, shift2angle
from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.mask import count, decode
from bloodmoon.mask import variance, snratio
from bloodmoon.io import simulation_files
from bloodmoon.optim import model_shadowgram, model_sky

import darksun as ds

**Analysis Methods**

In [2]:
#def _init_loss_metric(
#    true: NDArray,
#    pos: tuple[int, int],
#    camera: CodedMaskCamera,
#    model_source: Callable[[float, float, float], NDArray],
#) -> Callable[
#    [tuple[float, float, float]],
#    float,
#]:
#    """
#    Initialises the loss.
#    """
#    cropy, cropx = (
#        int(camera.specs.slit_deltay * camera.upscale_f.y / camera.specs.mask_deltay) + camera.upscale_f.y,
#        int(camera.specs.slit_deltax * camera.upscale_f.x / camera.specs.mask_deltax) + camera.upscale_f.x,
#    )
#    i, j = pos
#    slicey, slicex = (
#        slice(i - cropy, i + cropy + 1),
#        slice(j - cropx, j + cropx + 1),
#    )
#    true_ = true[slicey, slicex]
#
#    def f(args: tuple[float, float, float]) -> float:
#        """Loss metric for optimisation."""
#        sky = model_source(*args)[slicey, slicex]
#        metric_val = np.mean(np.square(sky - true_))
#        return metric_val
#    
#    return f


#def optimise(
#    true: NDArray,
#    arg_sky: tuple[int, int],
#    camera: CodedMaskCamera,
#    vignetting: bool,
#    psfy: bool,
#    verbose: bool = True,
#) -> tuple[float, float, float]:
#    """
#    Source parameters optimisation procedure.
#    """
#    px_dim_x, px_dim_y = (
#        camera.specs.mask_deltax / camera.upscale_f.x,
#        camera.specs.mask_deltay / camera.upscale_f.y,
#    )
#
#    source_model = _init_source_model(camera, vignetting, psfy)
#    loss = _init_loss_metric(true, arg_sky, camera, source_model)
#
#    sx_start, sy_start = pos2shift(camera, *arg_sky)
#    px_sx_start, px_sy_start = sx_start / px_dim_x, sy_start / px_dim_y
#    fluence_start = true[*arg_sky] / 0.9                 # camera coding power (Skinner et al. 2008)
#
#    with ds.timer('Optimising'):
#        results = least_squares(
#            loss,
#            x0=np.array((px_sx_start, px_sy_start, fluence_start)),
#            bounds=[
#                (
#                    max(px_sx_start - 3, -len(camera.bins_sky.x) // 2 + 1),
#                    max(px_sy_start - 3, -len(camera.bins_sky.y) // 2 + 1),
#                    true[*arg_sky],
#                ),
#                (
#                    min(px_sx_start + 3, len(camera.bins_sky.x) // 2 - 1),
#                    min(px_sy_start + 3, len(camera.bins_sky.y) // 2 - 1),
#                    true[*arg_sky] / 0.8,
#                ),
#            ],
#            xtol=1e-6,
#            ftol=1e-5,
#        )
#    # store the final optimized positions and fluence
#    px_sx, px_sy, fluence = map(float, results.x[:3])
#    sx, sy = px_sx * px_dim_x, px_sy * px_dim_y
#
#    # optimization verbose
#    if verbose:
#        print(
#            f'\n'
#            f'## Optimisation Results:\n'
#            f'  - fluence START: {fluence_start}\n'
#            f'  - shifts START (x, y): {sx_start}, {sy_start}\n'
#
#            f'  - fluence OPTIM.: {fluence}\n'
#            f'  - shifts OPTIM. (x, y): {sx}, {sy}\n'
#
#            f'  - fluence GAIN %: {(fluence - fluence_start) * 100 / fluence_start:.3f}\n'
#            f'  - shift_x GAIN %: {(sx - sx_start) * 100 / sx_start:.3f}\n'
#            f'  - shift_y GAIN %: {(sy - sy_start) * 100 / sy_start:.3f}\n'
#        )
#
#    return sx, sy, fluence

In [ ]:
from scipy.optimize import curve_fit


def process_skyimg(
    camera: CodedMaskCamera,
    sky: NDArray,
    pos: tuple[int, int],
) -> NDArray:
    """
    Processes the sky image for optimisation.
    """
    cropy, cropx = (
        int(camera.specs.slit_deltay * camera.upscale_f.y / camera.specs.mask_deltay) + 5,
        int(camera.specs.slit_deltax * camera.upscale_f.x / camera.specs.mask_deltax) + 7,
    )
    i, j = pos
    slicey, slicex = (
        slice(i - cropy, i + cropy + 1),
        slice(j - cropx, j + cropx + 1),
    )
    cropped = sky[slicey, slicex]
    return cropped.flatten()


def _ModelShiftFluence(
    camera: CodedMaskCamera,
    pos: tuple[int, int],
    vignetting: bool = True,
    psfy: bool = True,
) -> Callable[[NDArray, float, float, float], NDArray]:
    """
    A slow, vanilla implementation of the model for both direction and fluence optimization.
    Intended for debugging and benchmarking.

    Args:
        camera: CodedMaskCamera instance containing all geometric parameters
        pos: tuple of row, col indexes indicating the source peak position. The source
        sky image is cropped around `pos`.
        vignetting: If true, shadowgram model simulates vignetting.
        psfy: If true, the model used for optimization will simulate detector position
        reconstruction effects.

    Returns:
        A Callable, which is the routine for computing the model.
    """

    def f(x: NDArray, shift_x: float, shift_y: float, fluence: float) -> NDArray:
        """
        A simple, slow version of the model for both direction and fluence optimization.
        The input `x` represents an independent variable, and it has only been inserted
        to match the inputs of the scipy `curve_fit` procedure.

        Args:
            x: Placeholder for independent variable as in `curve_fit` doc
            shift_x: Source position x-coordinate in sky-shift space (mm)
            shift_y: Source position y-coordinate in sky-shift space (mm)
            fluence: Source intensity/fluence value

        Returns:
            Flattened and cropped 2D source-modeled sky image
        """
        modeled = model_sky(camera, shift_x, shift_y, fluence, vignetting, psfy)
        return process_skyimg(camera, modeled, pos)
    
    return f


def optimize(
    camera: CodedMaskCamera,
    sky: NDArray,
    arg_sky: tuple[int, int],
    vignetting: bool = True,
    psfy: bool = True,
    verbose: bool = True,
) -> tuple[float, float, float]:
    """
    Performs the optimization to fit a point source model to sky image data.

    This function performs the optimization by simultaneously fit the candidate
    position and fluence. The starting position is inferred from the candidate
    pixel position, while the starting fluence is represented by the counts at
    the candidate extracted pixel indexes.

    Args:
        camera: CodedMaskCamera instance containing detector and mask parameters
        sky: 2D array of the reconstructed sky image to fit
        arg_sky: Initial guess for source position as (row, col) indices
        vignetting: If true, the model used for optimization will simulate vignetting.
        psfy: If true, the model used for optimization will simulate detector position
        reconstruction effects.

    Returns:
        Tuple containing the best-fit parameters `(x, y, fluence)` where:
                - x, y are the optimized sky-shift coordinates
                - fluence is the optimized source intensity

    Notes:
        - Bounds are set based on initial guess and physical constraints
    """
    px_dim_x, px_dim_y = (
        camera.specs.mask_deltax / camera.upscale_f.x,
        camera.specs.mask_deltay / camera.upscale_f.y,
    )

    model_shift_flux = _ModelShiftFluence(camera, arg_sky, vignetting, psfy)
    sx_start, sy_start = pos2shift(camera, *arg_sky)
    sky_peak = sky[*arg_sky]
    fluence_start = (
        sky_peak / 0.85 if psfy else sky_peak
    )
    sky_ydata = process_skyimg(camera, sky, arg_sky)
    
    results, _ = curve_fit(
        model_shift_flux,
        xdata=np.arange(len(sky_ydata)),
        ydata=sky_ydata,
        p0=[sx_start, sy_start, fluence_start],
        bounds=[
            (
                max(sx_start - 1.5 * px_dim_x, camera.bins_sky.x[0]),
                max(sy_start - 1.5 * px_dim_y, camera.bins_sky.y[0]),
                sky_peak,
            ),
            (
                min(sx_start + 1.5 * px_dim_x, camera.bins_sky.x[-1]),
                min(sy_start + 1.5 * px_dim_y, camera.bins_sky.y[-1]),
                1.25 * sky_peak,
            ),
        ],
    )
    # store the final optimized positions and fluence
    sx, sy, fluence = map(float, results)

    if verbose:
        print(
            f'\n'
            f'## Optimisation Results:\n'
            f'  - fluence START: {fluence_start}\n'
            f'  - shifts START (x, y): {sx_start}, {sy_start}\n'

            f'  - fluence OPTIM.: {fluence}\n'
            f'  - shifts OPTIM. (x, y): {sx}, {sy}\n'

            f'  - fluence GAIN %: {(fluence - fluence_start) * 100 / fluence_start:.3f}\n'
            f'  - shift_x GAIN %: {np.sign(sx_start) * (sx - sx_start) * 100 / sx_start:.3f}\n'
            f'  - shift_y GAIN %: {np.sign(sy_start) * (sy - sy_start) * 100 / sy_start:.3f}\n'
        )

    return sx, sy, fluence

In [4]:
class Candidate(NamedTuple):
    """
    Source candidate main info container.

    Attributes:
        shift_x (float):
            Coded-mask camera local frame sky-coord along the x-axis [mm].
        shift_y (float):
            Coded-mask camera local frame sky-coord along the y-axis [mm].
        fluence (float):
            Observed candidate fluence [ph].
        snr (float):
            Candidate significance [adim].
    """
    shift_x: float
    shift_y: float
    fluence: float
    snr: float


def find_candidate(
    sky: NDArray,
    snr: NDArray,
    snr_threshold: int | float,
    batch: int = 1000,
) -> tuple[int, int] | bool:
    """
    Returns the position of a valid IROS candidate inside the sky image.
    """
    reservoir = np.array(
        [np.unravel_index(id_, sky.shape) for id_ in np.argsort(sky, axis=None)[-batch:]]
    )
    for pos in reservoir[::-1]:
        if (snr[*pos] > snr_threshold):
            return tuple(pos)
    return False


def init_fit_subtraction(
    camera: CodedMaskCamera,
    vignetting: bool,
    psfy: bool,
) -> tuple[
    Callable[[tuple[int, int], NDArray, NDArray], Candidate],
    Callable[[Candidate, NDArray], NDArray],
]:
    """
    Initialises the source params optimisation and source subtraction logics.
    """
    def fit_candidate_params(
        candidate_pos: tuple[int, int],
        sky: NDArray,
        snr: NDArray,
    ) -> Candidate:
        """Performs the optimisation of the source candidate params."""
        try:
            shift_x, shift_y, fluence = optimize(
                camera=camera,
                sky=sky,
                arg_sky=candidate_pos,
                vignetting=vignetting,
                psfy=psfy,
            )
        except Exception as e:
            raise RuntimeError(f"Optimization failed: {str(e)}") from e
        
        significance = float(snr[*candidate_pos])
        return Candidate(shift_x, shift_y, fluence, significance)

    def subtract(
        candidate: Candidate,
        detector: NDArray,
    ) -> NDArray:
        """Subtracts candidate from detector image."""
        sg_model = model_shadowgram(
            camera=camera,
            shift_x=candidate.shift_x,
            shift_y=candidate.shift_y,
            vignetting=vignetting,
            psfy=psfy,
        )
        residual = detector - candidate.fluence * sg_model
        return residual
    
    return fit_candidate_params, subtract

In [5]:
from astropy.io.fits.fitsrec import FITS_rec

from bloodmoon.types import CoordEquatorial
from bloodmoon.coords import equatorial2shift

from darksun.data import CatalogueLoader, DataLoader


def source_catalogue_data(
    sourceID: str,
    catalogue: CatalogueLoader,
) -> FITS_rec:
    """
    Extracts the source data from the catalogue.
    """
    data = (
        np.unique(
            catalogue.DLdata[(catalogue.DLdata['ID'] == sourceID)]
        )
    )[0]
    return data


def source_shifts(
    sourceID: str,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
) -> tuple[float, float]:
    """
    Extracts the source shifts coords from the catalogue.
    """
    data = source_catalogue_data(sourceID, catalogue)
    shiftx, shifty = equatorial2shift(sdl, camera, data['RA'], data['DEC'])
    return shiftx, shifty


def select_source_photons(
    coords: CoordEquatorial | tuple[CoordEquatorial, ...],
    data: FITS_rec,
    verbose: bool = True,
) -> FITS_rec:
    """
    Selects photon events relative to the input source RA/Dec coords.
    """
    mask = np.ones(len(data), dtype=bool)
    coords_ = (coords,) if isinstance(coords, CoordEquatorial) else coords
    for c in coords_:
        mask &= (
            (np.isclose(data['RA'], c.ra) & np.isclose(data['DEC'], c.dec))
        )
    selected = data[mask]
    if verbose:
        print(f'Selected {len(selected)}/{len(data)} photons.')
    return selected


def source_fluence(
    sourceID: str,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
    verbose: bool = True,
) -> float:
    """
    Extracts the source fluence from the catalogue.
    """
    data = source_catalogue_data(sourceID, catalogue)
    coords = CoordEquatorial(data['RA'], data['DEC'])
    total_photons = select_source_photons(coords, sdl.DLdata, verbose)
    # remove photons fell in detector plane dead zone
    det_image = count(camera, total_photons)[0] * (camera.bulk > 0)
    return det_image.sum()

In [6]:
def handle_det_spres(dataset: str) -> bool:
    """Handles detector spatial resolution correction."""
    if dataset not in ('detected', 'reconstructed'):
        raise ValueError('Nah-huh...')
    
    return False if dataset == 'detected' else True


def shift2arcmin(camera: CodedMaskCamera, shift: float) -> float:
    """Shift to angular coord in [arcmin] conversion."""
    return shift2angle(camera, shift) * 60


def fluence_error(observed: float, true: float) -> float:
    """Returns the percentage error on the observed fluence."""
    return (observed - true) * 100 / true


def optim_benchmark(
    true: tuple[float, float, float],
    candidate: Candidate,
    camera: CodedMaskCamera,
    camID: str,
    dataset: str,
) -> None:
    """
    Prints out optimisation results wrt the true values from catalogue.
    """
    sx, sy, f = true
    print(
        f'## {dataset.capitalize()} Dataset\n\n'

        f'# Fit {camID.upper()}\n'
        f'  - coords residues along fine dir: {shift2arcmin(camera, candidate.shift_x - sx)} [arcmin]\n'
        f'  - coords residues along coarse dir: {shift2arcmin(camera, candidate.shift_y - sy)} [arcmin]\n'
        f'  - fluence residues: {fluence_error(candidate.fluence, f)} [%]\n'
    )
    return None

**Mask and Data Specifics**

In [7]:
MASK_FITS: str = "wfm_mask_NTHT_20250725.fits"

SKYFIELD: str = "GalacticCentre"
DATA_FITS: str = "galctr_rxte-sax_2-50keV_mask_050_1040x17_opaquemask_infdet"

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "reconstructed"

UPS_X: int = 2
UPS_Y: int = 1

VIGNETTING: bool = True
PSFY: bool = handle_det_spres(DATASET)

**Load Mask and Data**

In [8]:
mask_path, simul_data, _ = _handle_dirpaths(
        mask=MASK_FITS,
        skyfield=SKYFIELD,
        simul=DATA_FITS,
    )
wfm: CodedMaskCamera = codedmask(mask_path, UPS_X, UPS_Y)
filepaths: dict[str, dict[str, Path]] = simulation_files(simul_data)

sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET])
catalogueA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])

sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET])
catalogueB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

In [9]:
fit_candidate_params, subtract = init_fit_subtraction(wfm, VIGNETTING, PSFY)

In [10]:
detector_camA = count(wfm, sdlA.DLdata)[0]
true_sky_camA = decode(wfm, detector_camA)
varmap_camA = variance(wfm, detector_camA)
snr_camA = snratio(true_sky_camA, varmap_camA)

## USING BULK MASK with 1.5 mm cover ##


In [11]:
detector_camB = count(wfm, sdlB.DLdata)[0]
true_sky_camB = decode(wfm, detector_camB)
varmap_camB = variance(wfm, detector_camB)
snr_camB = snratio(true_sky_camB, varmap_camB)

**Analysis for LEM-X Camera A**

In [12]:
arg_sky_camA = find_candidate(true_sky_camA, snr_camA, snr_threshold=5.0)
#sxA, syA, fA = optimize(wfm, true_sky_camA, arg_sky_camA, VIGNETTING, PSFY)
scox1 = fit_candidate_params(arg_sky_camA, true_sky_camA, snr_camA)

    # Starting 'Optimisation' at 14:37:21.
    # Finished 'Optimisation' in 00h:00m:3.314s.


## Optimisation Results:
  - fluence START: 1135364.2874728004
  - shifts START (x, y): 43.875, 78.0
  - fluence OPTIM.: 1052445.450820783
  - shifts OPTIM. (x, y): 43.91089183770604, 78.04458293946865
  - fluence GAIN %: -7.303
  - shift_x GAIN %: 0.082
  - shift_y GAIN %: 0.057



In [13]:
true_params_camA = (
    43.909402530749, 78.04581352417, (1064657.0 if DATASET == 'reconstructed' else 1073525.0),
)
optim_benchmark(true_params_camA, scox1, wfm, ID_CAMERA_A, DATASET)

## Reconstructed Dataset

# Fit CAM1A
  - coords residues along fine dir: 0.025214775583264223 [arcmin]
  - coords residues along coarse dir: -0.020834467289729316 [arcmin]
  - fluence residues: -1.1469937434513726 [%]



In [14]:
# iteration 2 for GX5-1
detector_gx51_camA = subtract(
    scox1,
    detector_camA,
)
sky_gx51_camA = decode(wfm, detector_gx51_camA)
snr_gx51_camA = snratio(sky_gx51_camA, varmap_camA)

arg_sky_gx51_camA = find_candidate(sky_gx51_camA, snr_gx51_camA, snr_threshold=5.0)
gx51 = fit_candidate_params(arg_sky_gx51_camA, sky_gx51_camA, snr_gx51_camA)

    # Starting 'Optimisation' at 14:37:25.
    # Finished 'Optimisation' in 00h:00m:3.360s.


## Optimisation Results:
  - fluence START: 120699.75443386675
  - shifts START (x, y): 13.5, -12.5
  - fluence OPTIM.: 112798.79677053036
  - shifts OPTIM. (x, y): 13.53426079829615, -12.431932385688429
  - fluence GAIN %: -6.546
  - shift_x GAIN %: 0.254
  - shift_y GAIN %: 0.545



In [15]:
true_params_gx51_camA = (
    *source_shifts('gx5-1', catalogueA, sdlA, wfm),
    source_fluence('gx5-1', catalogueA, sdlA, wfm),
)
optim_benchmark(true_params_gx51_camA, gx51, wfm, ID_CAMERA_A, DATASET)

Selected 123492/2564475 photons.
## Reconstructed Dataset

# Fit CAM1A
  - coords residues along fine dir: 0.08636476138641114 [arcmin]
  - coords residues along coarse dir: 1.3415125798935628 [arcmin]
  - fluence residues: -3.047173235809015 [%]



**Analysis for LEM-X Camera B**

In [16]:
arg_sky_camB = find_candidate(true_sky_camB, snr_camB, snr_threshold=5.0)
#sxB, syB, fB = optimize(wfm, true_sky_camB, arg_sky_camB, VIGNETTING, PSFY)
scox1_camB = fit_candidate_params(arg_sky_camB, true_sky_camB, snr_camB)

    # Starting 'Optimisation' at 14:37:28.
    # Finished 'Optimisation' in 00h:00m:3.884s.


## Optimisation Results:
  - fluence START: 1028076.4250290795
  - shifts START (x, y): 78.0, -44.0
  - fluence OPTIM.: 947321.8431214164
  - shifts OPTIM. (x, y): 78.04596118077806, -43.937703261718944
  - fluence GAIN %: -7.855
  - shift_x GAIN %: 0.059
  - shift_y GAIN %: 0.142



In [17]:
true_params_camB = (
    78.04581352417, -43.909402530749, (952281.0 if DATASET == 'reconstructed' else 955717.0),
)
optim_benchmark(true_params_camB, scox1_camB, wfm, ID_CAMERA_B, DATASET)

## Reconstructed Dataset

# Fit CAM1B
  - coords residues along fine dir: 0.0024999065627043585 [arcmin]
  - coords residues along coarse dir: -0.47914674155521036 [arcmin]
  - fluence residues: -0.5207661266562704 [%]



In [18]:
# iteration 2 for GX5-1
detector_gx51_camB = subtract(
    scox1_camB,
    detector_camB,
)
sky_gx51_camB = decode(wfm, detector_gx51_camB)
snr_gx51_camB = snratio(sky_gx51_camB, varmap_camB)

arg_sky_gx51_camB = find_candidate(sky_gx51_camB, snr_gx51_camB, snr_threshold=5.0)
gx51_camB = fit_candidate_params(arg_sky_gx51_camB, sky_gx51_camB, snr_gx51_camB)

    # Starting 'Optimisation' at 14:37:32.
    # Finished 'Optimisation' in 00h:00m:3.898s.


## Optimisation Results:
  - fluence START: 131392.26461590079
  - shifts START (x, y): -12.5, -13.5
  - fluence OPTIM.: 118711.65867556958
  - shifts OPTIM. (x, y): -12.508396337959349, -13.470865435563319
  - fluence GAIN %: -9.651
  - shift_x GAIN %: -0.067
  - shift_y GAIN %: 0.216



In [19]:
true_params_gx51_camB = (
    *source_shifts('gx5-1', catalogueB, sdlB, wfm),
    source_fluence('gx5-1', catalogueB, sdlB, wfm),
)
optim_benchmark(true_params_gx51_camB, gx51_camB, wfm, ID_CAMERA_B, DATASET)

Selected 123995/2493999 photons.
## Reconstructed Dataset

# Fit CAM1B
  - coords residues along fine dir: 0.046936410598879486 [arcmin]
  - coords residues along coarse dir: 0.9869531307980113 [arcmin]
  - fluence residues: 1.1181174248243813 [%]

